# handleMqttMessage Implementation in CTSensor

## Key Finding: CTSensor Does NOT Override handleMqttMessage()

### Inheritance Chain

```
Sensor.h (base class)
  ↓
CTSensor.h (derived class)
```

### In Sensor.h (Base Class)
Located at: `C:\Users\mcken\OneDrive\projects\iot\devCores\core_v2\Sensor.h`

```cpp
class Sensor {
public:
    virtual void setup() = 0;
    virtual void loop() = 0;
    virtual bool handleMqttMessage(const char* subtopic, const char* payload) {
        return false;  // Default implementation - does nothing
    }
};
```

The base class provides a **default implementation** that:
- Takes a subtopic and payload as parameters
- Always returns `false` (meaning "message not handled")
- Is marked `virtual` (can be overridden but doesn't have to be)

### In CTSensor.h
Located at: `C:\Users\mcken\OneDrive\projects\iot\gadgets\CTSensor\CTSensor.h`

**CTSensor does NOT override `handleMqttMessage()`**

This means:
- CTSensor inherits the default implementation from Sensor
- When `handleMqttMessage()` is called on a CTSensor object, it returns `false`
- CT sensors are **read-only** - they only publish data, they don't respond to MQTT commands

### What CTSensor Does Implement

CTSensor overrides these required methods:
1. **`setup()`** - Empty, no special setup needed
2. **`loop()`** - Main state machine for non-blocking sensor reading

### Why This Design?

- **CT sensors are passive** - they measure current and publish values
- **They don't need to respond to commands** - unlike controllable devices (relays, etc.)
- **The base class provides the interface** - other sensor types might override `handleMqttMessage()` if they need to handle commands

### If You Needed to Add MQTT Command Handling

You would add this to CTSensor class:

```cpp
bool handleMqttMessage(const char* subtopic, const char* payload) override {
    // Example: reset reading on command
    if (strcmp(subtopic, "reset") == 0) {
        _lastReportedValue = 0.0;
        return true;  // Message handled
    }
    return false;  // Message not recognized
}
```